In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Define paths
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data"
BREAD_PATH = DATA_PATH / "Bread"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(exist_ok=True)

MODELS_PATH = PROJECT_ROOT / "models"
MODELS_PATH.mkdir(exist_ok=True)

import pandas as pd
import xgboost as xgb
from src.models.utils import select_features
from src.utils.metrics import calculate_error_metrics

from src.models.registry import MODEL_REGISTRY
from src.models.evaluate import (load_model_and_test,
                                 plot_error_analysis,
                                 get_xgb_feature_importance,
                                 plot_xgb_feature_importance)
from src.features.encoding import (fit_gln_target_encoding,
                                   apply_gln_te,
                                   fit_gb_id_target_encoding,
                                   apply_gb_id_target_encoding)
from src.utils.splitting import split_by_time

In [ ]:
store_daily = pd.read_parquet(PROCESSED_PATH / "store_daily.parquet")
store_daily["date"] = pd.to_datetime(store_daily["date"])

store_daily.head()

In [ ]:
test_start = store_daily["date"].max() - pd.DateOffset(days=365)
test_end = store_daily["date"].max()

train_df, test_df = split_by_time(store_daily, test_start, test_end)

print(train_df.shape, test_df.shape)

In [ ]:
store_daily.columns

In [ ]:
# target encoding for gln
mapping, global_mean = fit_gln_target_encoding(train_df)

train_df = apply_gln_te(train_df, mapping, global_mean)
test_df = apply_gln_te(test_df, mapping, global_mean)

***XGBoost Model (total)***

In [ ]:
MODEL_TYPE = "xgb_total"  # or "rf"
trainer = MODEL_REGISTRY[MODEL_TYPE]

results = trainer(
    train_df,
    model_path=MODELS_PATH / f"{MODEL_TYPE}.pkl"
)

In [ ]:
test_results = load_model_and_test(
    test_df,
    model_path=MODELS_PATH / f"{MODEL_TYPE}.pkl"
)

plot_error_analysis(
    test_results["y_test"],
    test_results["y_pred"]
)

test_results["metrics"]

In [ ]:
model = test_results["model"]

imp_df = get_xgb_feature_importance(model, feature_names=train_df.columns)

plot_xgb_feature_importance(imp_df, title="Total XGB – Feature Importance (Gain)")

### Feature-Pruning Quick Compare (XGB Total)
Use this to compare baseline vs top-k important features.

In [ ]:
# Baseline features (after select_features, includes minimal de-duplication)
X_base, y_base = select_features(train_df.copy(), "xgb_total")

DEFAULT_PARAMS = {
    "random_state": 42,
    "objective": "reg:squarederror",
    "tree_method": "hist",
}

# If you already ran Optuna above, reuse best_params automatically.
params = DEFAULT_PARAMS.copy()
try:
    params.update(best_params)  # if defined explicitly
except NameError:
    pass
try:
    params.update(results["best_params"])  # from trainer output
except Exception:
    pass

baseline_model = xgb.XGBRegressor(**params)
baseline_model.fit(X_base, y_base)

X_test_base = test_df[X_base.columns]
y_test = test_df["quantity"].values
y_pred_base = baseline_model.predict(X_test_base)
metrics_base = calculate_error_metrics(y_test, y_pred_base, verbose=False)

imp_df = get_xgb_feature_importance(baseline_model, feature_names=X_base.columns)
TOP_K = 20  # adjust
IMPORTANCE_THRESHOLD = None  # e.g., 0.01 to drop low-importance features

if IMPORTANCE_THRESHOLD is not None:
    keep_cols = imp_df[imp_df["importance"] > IMPORTANCE_THRESHOLD]["feature"].tolist()
    mode = f"threshold>{IMPORTANCE_THRESHOLD}"
else:
    keep_cols = imp_df["feature"].head(TOP_K).tolist()
    mode = f"top_k={TOP_K}"

if not keep_cols:
    raise ValueError("No features selected. Lower threshold or increase TOP_K.")

X_pruned = X_base[keep_cols]
pruned_model = xgb.XGBRegressor(**params)
pruned_model.fit(X_pruned, y_base)

X_test_pruned = test_df[keep_cols]
y_pred_pruned = pruned_model.predict(X_test_pruned)
metrics_pruned = calculate_error_metrics(y_test, y_pred_pruned, verbose=False)

summary = pd.DataFrame(
    [metrics_base, metrics_pruned],
    index=["baseline", "pruned"],
).round(4)

print(f"Selection mode: {mode}")
print(f"Baseline feature count: {X_base.shape[1]}")
print(f"Pruned feature count: {len(keep_cols)}")
print("Kept features:")
print(pd.Series(keep_cols))

summary

### Train and evaluate share model

In [ ]:
share_df = pd.read_parquet(PROCESSED_PATH / "share_daily.parquet")
share_train, share_test = split_by_time(share_df, test_start=test_start, test_end=test_end)

In [ ]:
share_df.columns

In [ ]:
# gb_id target encoding
mapping_gb, global_mean_gb = fit_gb_id_target_encoding(share_train)

share_train = apply_gb_id_target_encoding(share_train, mapping_gb, global_mean_gb)
share_test = apply_gb_id_target_encoding(share_test, mapping_gb, global_mean_gb, )

In [ ]:
# target encoding for gln
mapping, global_mean = fit_gln_target_encoding(share_train)

share_train = apply_gln_te(share_train, mapping, global_mean)
share_test = apply_gln_te(share_test, mapping, global_mean)

In [ ]:
share_train["promo_x_popularity"] = share_train["on_promotion"] * share_train["gb_id_mean_target"]
share_test["promo_x_popularity"] = share_test["on_promotion"] * share_test["gb_id_mean_target"]

In [ ]:
MODEL_TYPE = "xgb_share"
trainer = MODEL_REGISTRY[MODEL_TYPE]

results = trainer(share_train, model_path=MODELS_PATH / f"{MODEL_TYPE}.pkl")
test_results = load_model_and_test(share_test, model_path=MODELS_PATH / f"{MODEL_TYPE}.pkl")

plot_error_analysis(test_results["y_test"], test_results["y_pred"])

test_results["metrics"]

## Direct model

In [ ]:
direct_df = pd.read_parquet(PROCESSED_PATH / "direct_daily.parquet")
direct_train, direct_test = split_by_time(direct_df, test_start=test_start, test_end=test_end)

In [ ]:
direct_df.columns

In [ ]:
# gb_id target encoding
mapping_gb, global_mean_gb = fit_gb_id_target_encoding(direct_train)

direct_train = apply_gb_id_target_encoding(direct_train, mapping_gb, global_mean_gb)
direct_test = apply_gb_id_target_encoding(direct_test, mapping_gb, global_mean_gb)

In [ ]:
# target encoding for gln
mapping, global_mean = fit_gln_target_encoding(direct_train)

direct_train = apply_gln_te(direct_train, mapping, global_mean)
direct_test = apply_gln_te(direct_test, mapping, global_mean)

In [ ]:
direct_train["promo_x_popularity"] = direct_train["on_promotion"] * direct_train["gb_id_mean_target"]
direct_test["promo_x_popularity"] = direct_test["on_promotion"] * direct_test["gb_id_mean_target"]

In [ ]:
MODEL_TYPE = "xgb_direct"
trainer = MODEL_REGISTRY[MODEL_TYPE]

results = trainer(direct_train, model_path=MODELS_PATH / f"{MODEL_TYPE}.pkl")

test_results = load_model_and_test(direct_test, model_path=MODELS_PATH / f"{MODEL_TYPE}.pkl")

plot_error_analysis(test_results["y_test"], test_results["y_pred"])

test_results["metrics"]

In [ ]:
model = test_results["model"]

imp_df = get_xgb_feature_importance(model, feature_names=direct_train.drop(columns=["target"]).columns)

plot_xgb_feature_importance(imp_df, title="Direct XGB – Feature Importance (Gain)")

In [ ]:
imp_df